# TabFM Financial Risk Benchmark -- Colab runner

Runs the parts of [`tabfm-financial-risk-benchmark`](https://github.com/ConceptualCode/tabfm-financial-risk-benchmark) that need more RAM/GPU than a typical laptop has:

- **TabFM**: ~6.5GB checkpoint, needs a real GPU to load efficiently (see `models.py::build_tabfm`'s meta-device loader). Its memory footprint scales with **both** ensemble size (`n_estimators`) **and** dataset width (feature count) -- confirmed by a real CUDA OOM on an 8GB GPU for a 23-column dataset (`cd2`) at default settings, even outside the SHAP step. Colab's 16GB T4 has roughly double that headroom, but very wide datasets (e.g. `cf2` at 120 columns) are untested and could still be tight -- see step 4's fallback if a dataset OOMs on the main metrics step, not just SHAP.
- **SAP-RPT** (`sap-rpt-1-oss`): tiny weights (~65MB) but its default settings (`bagging=8`, `max_context_size=8192`) can need up to ~80GB of GPU memory for inference-time activations. We constrain both via env vars below to fit a free-tier T4 (16GB VRAM).

**Before running:** go to Runtime -> Change runtime type -> select a GPU (T4 is fine, free tier).

**HF access:** `sap_rpt` requires requesting access at https://huggingface.co/SAP/sap-rpt-1-oss first. If you don't have access yet, skip the `sap_rpt` entries in the run cells below -- everything else still works.

## 1. Sanity-check the runtime (GPU + RAM)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h

## 2. Clone the repo and install dependencies

Colab ships Python 3.11/3.12 already, so no Python-version workaround is needed here (unlike the laptop issue we hit locally, which required upgrading from 3.10).

In [ ]:
!python --version
!git clone https://github.com/ConceptualCode/tabfm-financial-risk-benchmark.git
%cd tabfm-financial-risk-benchmark

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. HuggingFace auth (needed only for `sap_rpt`)

Add your token via Colab's Secrets manager (key icon in the left sidebar) as `HF_TOKEN` -- do **not** paste it directly into a cell, since that embeds it in the notebook file if you ever save/share it.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')  # requires the HF_TOKEN secret to be set, see markdown above
login(token=hf_token)

## 4. Constrain ensemble sizes to fit the T4 and speed up test runs

SAP-RPT's default settings (`bagging=8`, `max_context_size=8192`) target ~80GB GPUs -- these reduced values are a starting point, push them up if you have headroom, or down further if you still hit an OOM.

TabFM's `TabFMClassifier` defaults to a 32-member ensemble (`n_estimators=32`) -- every prediction actually runs the model 32 times, which is what dominates TabFM's ~4-minute-per-dataset cost on a T4 (separate from the one-time 6.5GB checkpoint download). Lower `TABFM_N_ESTIMATORS` for quick validation runs; leave it at the library default (32) for the full benchmark, since that's the setting the real accuracy/calibration numbers should reflect.

**If a dataset OOMs on TabFM's *main metrics* step** (not the SHAP step -- watch the error message, `FAILED: <dataset>/tabfm` means the main step, `SHAP FAILED: <dataset>/tabfm` means only the explanation step failed): that dataset is wider than the T4 can handle at `n_estimators=32`. The per-dataset/model exception handling already logs it and moves on to the next model/dataset automatically -- you don't need to restart anything. To recover that specific dataset's TabFM result, rerun just it with a lower ensemble size and accept the fidelity tradeoff (smaller ensemble = less robust estimate, but still a real result):
```python
os.environ['TABFM_N_ESTIMATORS'] = '8'  # or lower
```
then `!python scripts/run_benchmark.py --datasets cd2 --models tabfm` (swap in whichever dataset OOM'd). Reset back to `32` afterward for any remaining datasets.

**SHAP-specific settings** (`TABFM_SHAP_N_ESTIMATORS`, `SAP_RPT_SHAP_BAGGING`): running SHAP's `KernelExplainer` on top of a *full-settings* TabFM/SAP-RPT instance is a different problem from the main metrics -- a single full-ensemble TabFM instance already uses ~11-12GB of a 16GB T4 just sitting fitted, leaving too little headroom for `KernelExplainer`'s repeated perturbation passes (confirmed: real CUDA OOM in testing), and SAP-RPT's SHAP step scales directly with bagging (~59 min for one dataset at `bagging=2`). The benchmark script automatically frees the full-settings instance and builds a separate, smaller one just for SHAP, controlled by these two env vars -- the real accuracy/calibration/fairness numbers still come from the full-settings model above, unaffected.

In [ ]:
import os
os.environ['SAP_RPT_BAGGING'] = '2'
os.environ['SAP_RPT_MAX_CONTEXT_SIZE'] = '1024'
os.environ['TABFM_N_ESTIMATORS'] = '4'  # lower for fast validation runs; unset/remove for the full benchmark
os.environ['TABFM_SHAP_N_ESTIMATORS'] = '4'  # separate, smaller ensemble just for the SHAP step
os.environ['SAP_RPT_SHAP_BAGGING'] = '1'  # separate, smaller bagging just for the SHAP step

## 5. Mount Google Drive so results survive a session disconnect (recommended)

The benchmark script writes results incrementally to `results/*.jsonl` as soon as each (dataset, model) pair finishes -- but that still lands on the Colab VM's local disk by default, which is wiped the moment the session disconnects. Colab free-tier sessions *do* disconnect unpredictably (idle timeout, max runtime, a closed tab), and the full grid (10 datasets x 5 models, with SHAP) can run long enough for that to matter.

Setting `TABFM_RESULTS_DIR` to a Drive-mounted path makes every incremental write go straight to your Drive in real time, so nothing is lost even if the session dies mid-run and you never get to the download cell at the end. Skip this cell if you'd rather just rely on the manual download step later -- everything still works, you just risk losing an in-progress run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['TABFM_RESULTS_DIR'] = '/content/drive/MyDrive/tabfm_results'
os.makedirs(os.environ['TABFM_RESULTS_DIR'], exist_ok=True)

## 6. Smoke tests -- confirm the base setup works before touching the foundation models

In [ ]:
!python -m pytest tests/test_smoke.py -v

## 7. Validation run -- one dataset, skip SHAP

Confirms TabFM and SAP-RPT actually load and predict end to end before committing to the full grid. Drop `sap_rpt` from `--models` if you don't have HF access yet. Watch for `[tabfm] using device: cuda` / `[sap_rpt] using device: cuda` in the output -- if either says `cpu`, something's wrong with the GPU runtime, and this will be very slow.

In [ ]:
!python scripts/run_benchmark.py --datasets cd1 --models tabfm sap_rpt xgboost --skip-shap

## 8. Full benchmark

All 10 FinBench datasets, all models, including SHAP, cross-model prediction agreement, and the fairness audit (on the configs with a protected attribute). This is the long-running step -- Colab free-tier sessions can disconnect after ~12 hours idle/max runtime, but results are now written incrementally (one row appended to `results/*.jsonl` as soon as it's computed, not held in memory until the end), so a disconnect only costs the remaining unfinished work.

**Before running this for real numbers**, reset `TABFM_N_ESTIMATORS` back to the library default (32) if you lowered it for validation in step 4 -- otherwise the accuracy/calibration results won't reflect TabFM's intended configuration:
```python
os.environ.pop('TABFM_N_ESTIMATORS', None)
```
`TABFM_SHAP_N_ESTIMATORS`/`SAP_RPT_SHAP_BAGGING` are a different knob -- they only affect the SHAP explanation step, not the real metrics, so they're fine to leave small even for the full benchmark. If TabFM's SHAP still OOMs at the default `TABFM_SHAP_N_ESTIMATORS=4`, try `2` or `1`; if SAP-RPT's SHAP is still too slow at `SAP_RPT_SHAP_BAGGING=1`, that's already the minimum, so consider running SHAP for `tabfm`/`sap_rpt` on fewer datasets separately (e.g. `--datasets cd1 cf1 --models tabfm sap_rpt`) rather than as part of the full 10-dataset grid.

**If TabFM OOMs on the main metrics step for a specific (wide) dataset**, see step 4's fallback above -- rerun just that dataset with a lower `TABFM_N_ESTIMATORS`.

If the session *does* disconnect partway through: reconnect, re-run cells 1-6 (and cell 5 if you mounted Drive, to remount it), then re-run this cell -- it appends to the existing `results/*.jsonl` by default, so already-completed (dataset, model) pairs just get duplicated rows rather than lost work. To skip straight to re-exporting CSVs from whatever's already saved without re-running any models, use `--export-only` instead. To intentionally start over from empty results, add `--fresh`.

In [ ]:
!python scripts/run_benchmark.py

## 9. Missing-data robustness sweep (RQ7)

Trains each model normally, then evaluates it against the same test set with 0%/5%/20%/50% of features masked to missing -- tests whether TabFM/SAP-RPT's advertised "automatic missing-data handling" actually degrades gracefully, same question for XGBoost/LightGBM's native handling. Independent of the SHAP settings above; writes to `results/robustness.csv`.

In [ ]:
!python scripts/run_robustness.py

## 10. Pull the results back down

If you mounted Drive in step 5, your results already live at `/content/drive/MyDrive/tabfm_results/` -- this step is optional. Otherwise, this zips the VM-local `results/` and downloads it, so you can drop it into your local clone of the repo and commit from there.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('results', 'zip', 'results')
files.download('results.zip')